# Demo 06 : Composition and durability

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · composing-workflows

We port the compiled-subgraph shape from ARI510 and compose lookup and report workflows. Shared state keys pass through directly. A wrapper maps state when a child has a different schema.

In [2]:
import json
import operator
import sqlite3
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Usage(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for batch in response.generations:
            for generation in batch:
                usage = generation.message.usage_metadata or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)

usage = Usage()
llm = make_llm(callbacks=[usage], max_retries=0, max_tokens=650)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
    conn.row_factory = sqlite3.Row
    work_order = dict(conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", ("BW-7001",)).fetchone())
    part = dict(conn.execute("SELECT * FROM parts WHERE part_no = ?", ("CE-CF-02",)).fetchone())
print("work order:", work_order)
print("part:", part)

def config(label):
    return {"configurable": {"thread_id": f"07-{label}-{uuid4()}"}, "recursion_limit": 30}

work order: {'wo_id': 'BW-7001', 'fleet_no': 'CE12-2021-0117', 'opened': '2026-09-14', 'status': 'open', 'error_code': 'F-CE-110', 'summary': 'Traction system over-temperature during afternoon route', 'mechanic_id': 'M01'}
part: {'part_no': 'CE-CF-02', 'model': 'CE-12', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}


In [3]:
class Work(TypedDict):
    wo_id: str
    facts: str
    report: str
    queue: str
    status: str

def lookup(state):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", (state["wo_id"],)).fetchone()
    if row is None:
        raise ValueError("Unknown work order")
    return {"facts": json.dumps(dict(row))}

lookup_builder = StateGraph(Work)
lookup_builder.add_node("lookup", lookup)
lookup_builder.add_edge(START, "lookup")
lookup_builder.add_edge("lookup", END)
lookup_workflow = lookup_builder.compile()

class ReportInput(TypedDict):
    source: str
    text: str

def report(state):
    return {"text": llm.invoke(f"Summarise this Veenhaven Bus work order in one short sentence. Do not add facts: {state['source']}").content}

report_builder = StateGraph(ReportInput)
report_builder.add_node("report", report)
report_builder.add_edge(START, "report")
report_builder.add_edge("report", END)
report_workflow = report_builder.compile()

def mapped_report(state):
    child = report_workflow.invoke({"source": state["facts"], "text": ""})
    return {"report": child["text"]}

composed = StateGraph(Work)
composed.add_node("lookup_workflow", lookup_workflow)
composed.add_node("report_workflow", mapped_report)
composed.add_edge(START, "lookup_workflow")
composed.add_edge("lookup_workflow", "report_workflow")
composed.add_edge("report_workflow", END)
initial = {"wo_id": work_order["wo_id"], "facts": "", "report": "", "queue": "", "status": ""}
out = composed.compile().invoke(initial)
print("facts:", out["facts"])
print("report:", out["report"])
assert out["facts"] and out["report"]

facts: {"wo_id": "BW-7001", "fleet_no": "CE12-2021-0117", "opened": "2026-09-14", "status": "open", "error_code": "F-CE-110", "summary": "Traction system over-temperature during afternoon route", "mechanic_id": "M01"}
report: Veenhaven Bus work order BW-7001 is open for a traction system over-temperature issue reported during an afternoon route.


## b · branching-at-scale

We keep a routing registry separate from node implementations. Business rules route validated fault codes to a specialist queue or manual review. Unknown codes use an explicit default path.

In [4]:
from functools import partial

routes = {"F-CE": "electric", "F-MM": "diesel", "F-CD": "doors"}
def classify(state):
    code = json.loads(state["facts"])["error_code"]
    return {"queue": routes.get(code[:4], "manual")}

def queue_work(state, queue):
    return {"status": f"queued for {queue}"}

branching = StateGraph(Work)
branching.add_node("lookup", lookup_workflow)
branching.add_node("classify", classify)
for name in sorted(set(routes.values()) | {"manual"}):
    branching.add_node(name, partial(queue_work, queue=name))
    branching.add_edge(name, END)
branching.add_edge(START, "lookup")
branching.add_edge("lookup", "classify")
path_map = {name: name for name in set(routes.values()) | {"manual"}}
branching.add_conditional_edges("classify", lambda s: s["queue"], path_map)
routed = branching.compile().invoke(initial)
print("route:", routed["queue"], routed["status"])
assert routed["queue"] == "electric"
fallback_input = {**initial, "facts": json.dumps({"error_code": "UNKNOWN"})}
print("unknown route:", classify(fallback_input))
assert classify(fallback_input)["queue"] == "manual"

route: electric queued for electric
unknown route: {'queue': 'manual'}


## c · resuming-after-a-crash

We save lookup and model output before a node raises an injected crash. We close the SQLite connection and rebuild the graph, then resume the same thread with `None`. An approval interrupt instead requires `Command(resume=...)`.

In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.errors import GraphDrained
from langgraph.runtime import RunControl, Runtime

crash = {"enabled": True}
drain_request = {"enabled": False}
executions = {"prepare": 0, "finish": 0}

def prepare(state, runtime: Runtime):
    executions["prepare"] += 1
    update = lookup(state)
    update["report"] = llm.invoke(f"Draft a short Veenhaven Bus inspection note. Use only: {update['facts']}").content
    if drain_request["enabled"] and runtime.control is not None:
        runtime.control.request_drain("trainer shutdown")
    return update

def finish(state):
    executions["finish"] += 1
    if crash["enabled"]:
        raise RuntimeError("injected worker crash before recording")
    return {"status": "inspection note recorded"}

def durable_builder():
    graph = StateGraph(Work)
    graph.add_node("prepare", prepare)
    graph.add_node("finish", finish)
    graph.add_edge(START, "prepare")
    graph.add_edge("prepare", "finish")
    graph.add_edge("finish", END)
    return graph

checkpoint_file = Path("workflow-checkpoints.sqlite")
if checkpoint_file.exists():
    checkpoint_file.unlink()
crash_cfg = config("crash")
calls_before = usage.calls
with SqliteSaver.from_conn_string(str(checkpoint_file)) as saver:
    running = durable_builder().compile(checkpointer=saver)
    try:
        running.invoke(initial, crash_cfg, durability="sync")
    except RuntimeError as exc:
        print("crash:", str(exc))
    saved = running.get_state(crash_cfg)
    print("saved next:", saved.next)
    print("saved report:", saved.values["report"])
    assert saved.next == ("finish",) and saved.values["report"]
    assert executions == {"prepare": 1, "finish": 1}

crash["enabled"] = False
with SqliteSaver.from_conn_string(str(checkpoint_file)) as saver:
    restarted = durable_builder().compile(checkpointer=saver)
    print("reopened next:", restarted.get_state(crash_cfg).next)
    recovered = restarted.invoke(None, crash_cfg, durability="sync")
    print("recovered:", recovered["status"])
    print("executions:", executions)
    print("model calls:", usage.calls - calls_before)
    assert recovered["status"] == "inspection note recorded"
    assert executions == {"prepare": 1, "finish": 2}
    assert usage.calls - calls_before == 1

crash: injected worker crash before recording
saved next: ('finish',)
saved report: **Veenhaven Bus Inspection Note**

- **Work order:** BW-7001  
- **Fleet no.:** CE12-2021-0117  
- **Opened:** 2026-09-14  
- **Status:** Open  
- **Error code:** F-CE-110  
- **Summary:** Traction system over-temperature during afternoon route  
- **Mechanic:** M01
reopened next: ('finish',)
recovered: inspection note recorded
executions: {'prepare': 1, 'finish': 2}
model calls: 1


We also request a graceful drain inside the running preparation node. The node completes and its update is saved before the next node starts. A production signal handler can request the same drain from another thread.

In [6]:
drain_request["enabled"] = True
drain_cfg = config("drain")
control = RunControl()
before = dict(executions)
with SqliteSaver.from_conn_string(str(checkpoint_file)) as saver:
    running = durable_builder().compile(checkpointer=saver)
    try:
        running.invoke(initial, drain_cfg, control=control, durability="sync")
    except GraphDrained as exc:
        print("drained:", exc.reason)
    print("drained next:", running.get_state(drain_cfg).next)
    assert running.get_state(drain_cfg).next == ("finish",)
    assert executions["finish"] == before["finish"]
with SqliteSaver.from_conn_string(str(checkpoint_file)) as saver:
    restarted = durable_builder().compile(checkpointer=saver)
    resumed = restarted.invoke(None, drain_cfg, control=RunControl(), durability="sync")
    print("resumed:", resumed["status"])
    assert resumed["status"] == "inspection note recorded"

drained: trainer shutdown
drained next: ('finish',)
resumed: inspection note recorded


We use a new control object for the resumed run. Drain is cooperative and does not cancel in-flight work. A checkpoint cannot undo external writes; recording a real reservation still requires a stable idempotency key and a persistent backend.

In [7]:
print("metrics:", json.dumps({"model": MODEL, "calls": usage.calls,
      "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
      "seconds": round(time.perf_counter() - started, 1)}))

metrics: {"model": "openai/gpt-6-luna", "calls": 3, "input_tokens": 286, "output_tokens": 183, "seconds": 8.9}


## Things to try

Optional : we can crash before preparation completes and compare the repeated model calls with the crash after its checkpoint.